# Kaggle Titanic：从数据检查到可提交模型

这个 notebook 是一条可从空内核顺序运行的教学路线。它不会使用外部幸存者名单，也不会把 `gender_submission.csv` 当成答案。最终目标是生成经过格式验证的 `outputs/submission.csv`。

## Goal

完成一次规范的表格二分类实验：

1. 验证官方数据的结构与质量；
2. 探索训练样本中的生还关联；
3. 在 Pipeline 内完成特征工程、缺失值处理和编码；
4. 用开发集五折交叉验证比较模型；
5. 只使用一次保留验证集评估最终候选；
6. 使用完整训练集重新拟合并生成 Kaggle 提交文件。

**判断边界：** EDA 中看到的是这个训练样本里的统计关联，不是因果关系，也不能直接当作官方测试集事实。

## Setup

运行前请从 [Kaggle 官方数据页](https://www.kaggle.com/competitions/titanic/data) 下载 `train.csv`、`test.csv` 和 `gender_submission.csv`，放进项目的 `data/raw/`。如果文件缺失，下面的检查会明确停止，而不是改用其他来源。

In [ ]:
from pathlib import Path
import os
import sys
import warnings

PROJECT_ROOT = next(
    (
        candidate.resolve()
        for candidate in (Path.cwd(), Path.cwd().parent)
        if (candidate / "src").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("请从项目根目录或 notebooks/ 目录启动这个 notebook。")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = Path(
    os.environ.get("TITANIC_DATA_DIR", PROJECT_ROOT / "data" / "raw")
).resolve()
OUTPUT_DIR = Path(
    os.environ.get("TITANIC_OUTPUT_DIR", PROJECT_ROOT / "outputs")
).resolve()
MODEL_DIR = Path(
    os.environ.get("TITANIC_MODEL_DIR", PROJECT_ROOT / "models")
).resolve()
RANDOM_STATE = 42

# Keep library caches inside the project so restricted environments do not
# need to write to the user profile.
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".cache" / "matplotlib"))
(PROJECT_ROOT / ".cache" / "matplotlib").mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore", category=FutureWarning)
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedGroupKFold,
    cross_val_score,
    train_test_split,
)

from src.titanic_pipeline import (
    make_model_candidates,
    make_pipeline,
    validate_raw_data,
    validate_submission,
)

pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid", context="notebook")

In [ ]:
required_files = {
    "train": DATA_DIR / "train.csv",
    "test": DATA_DIR / "test.csv",
    "sample_submission": DATA_DIR / "gender_submission.csv",
}
missing_files = [str(path) for path in required_files.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "缺少 Kaggle 官方数据文件：\n- "
        + "\n- ".join(missing_files)
        + "\n请登录 Kaggle、接受比赛规则并下载后再运行。"
    )

train = pd.read_csv(required_files["train"])
test = pd.read_csv(required_files["test"])
sample_submission = pd.read_csv(required_files["sample_submission"])

validate_raw_data(train, test, strict_competition_shape=True)
print(f"train shape: {train.shape}; test shape: {test.shape}")
display(train.head())

## Steps

### 1. Validate and understand the data

先检查字段类型、缺失比例和 ID 唯一性。`Survived` 只允许出现在训练集；测试集没有真实标签。

In [ ]:
schema_summary = pd.DataFrame(
    {
        "dtype": train.dtypes.astype(str),
        "missing_count": train.isna().sum(),
        "missing_pct": (train.isna().mean() * 100).round(2),
        "unique_count": train.nunique(dropna=False),
    }
).sort_values("missing_pct", ascending=False)

assert train["PassengerId"].is_unique
assert test["PassengerId"].is_unique
assert "Survived" not in test.columns
assert set(train["Survived"].unique()) == {0, 1}

display(schema_summary)

### 2. Exploratory data analysis

先查看总体生还率和关键类别的组内生还率，再看缺失值、年龄和票价分布。下面所有数字都只描述 `train.csv` 样本。

In [ ]:
overall_survival_rate = train["Survived"].mean()
print(f"训练样本总体生还率: {overall_survival_rate:.3f}")

eda = train.copy()
eda["FamilySize"] = eda["SibSp"] + eda["Parch"] + 1
eda["IsAlone"] = np.where(eda["FamilySize"] == 1, "Alone", "With family")

group_tables = []
for column in ["Sex", "Pclass", "Embarked", "IsAlone"]:
    summary = (
        eda.groupby(column, dropna=False, observed=False)["Survived"]
        .agg(passengers="size", survival_rate="mean")
        .reset_index()
    )
    summary.insert(0, "feature", column)
    summary = summary.rename(columns={column: "group"})
    group_tables.append(summary)

survival_by_group = pd.concat(group_tables, ignore_index=True)
survival_by_group["survival_rate"] = survival_by_group["survival_rate"].round(3)
display(survival_by_group)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
missing_pct = train.isna().mean().mul(100).sort_values(ascending=False)
sns.barplot(x=missing_pct.values, y=missing_pct.index, ax=axes[0], color="#4C78A8")
axes[0].set(title="Missing values in train.csv", xlabel="Missing (%)", ylabel="")

sns.countplot(data=train, x="Survived", ax=axes[1], color="#72B7B2")
axes[1].set(title="Training target counts", xlabel="Survived", ylabel="Passengers")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for column, axis in zip(["Sex", "Pclass", "Embarked", "IsAlone"], axes.flat):
    sns.barplot(data=eda, x=column, y="Survived", errorbar=None, ax=axis, color="#F58518")
    axis.set(title=f"Sample survival rate by {column}", ylabel="Survival rate")
    axis.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(
    data=train,
    x="Age",
    hue="Survived",
    bins=25,
    element="step",
    stat="density",
    common_norm=False,
    ax=axes[0],
)
axes[0].set_title("Age distribution by outcome")

sns.histplot(
    data=train,
    x="Fare",
    hue="Survived",
    bins=30,
    element="step",
    stat="density",
    common_norm=False,
    ax=axes[1],
)
axes[1].set_xlim(0, train["Fare"].quantile(0.98))
axes[1].set_title("Fare distribution by outcome (up to 98th percentile)")
plt.tight_layout()
plt.show()

### 3. Split by family groups

先根据姓名中的姓氏和 `FamilySize` 建立一个用于验证的家庭组编号。单独出行者使用自己的 `PassengerId`，避免把同姓的陌生人误当成一家人。

切分时，同一个家庭组只能完整地出现在训练或验证的一边。家庭组编号只控制如何分组考试，**不会作为模型输入特征**。先用一折作为约 20% 的最终保留集；剩余开发集再做 5 折家庭分组交叉验证。

In [ ]:
X = train.drop(columns="Survived")
y = train["Survived"].astype(int)

def make_family_groups(frame):
    surname = (
        frame["Name"]
        .str.split(",", n=1)
        .str[0]
        .str.strip()
        .str.lower()
    )
    family_size = frame["SibSp"] + frame["Parch"] + 1
    family_group = surname + "_" + family_size.astype(str)
    singleton_group = "solo_" + frame["PassengerId"].astype(str)
    return family_group.where(family_size > 1, singleton_group)

all_family_groups = make_family_groups(X)

# The first group-aware fold becomes the sealed holdout (about 20%).
outer_group_split = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)
development_index, holdout_index = next(
    outer_group_split.split(X, y, groups=all_family_groups)
)

X_development = X.iloc[development_index].copy()
X_holdout = X.iloc[holdout_index].copy()
y_development = y.iloc[development_index].copy()
y_holdout = y.iloc[holdout_index].copy()
groups_development = all_family_groups.iloc[development_index].copy()
groups_holdout = all_family_groups.iloc[holdout_index].copy()

assert set(groups_development).isdisjoint(set(groups_holdout))

cross_validation = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print(f"Development rows: {len(X_development)}")
print(f"Untouched holdout rows: {len(X_holdout)}")
print(f"Development family groups: {groups_development.nunique()}")
print(f"Holdout family groups: {groups_holdout.nunique()}")
print("Family overlap: 0 groups")

### 4. Compare bounded model candidates with family-group validation

每个候选模型使用相同的家庭分组折。同一家人不会同时出现在训练折和验证折，因此这里测试的是模型面对未见家庭时的表现。模型和输入特征保持不变。

In [ ]:
candidates = make_model_candidates(random_state=RANDOM_STATE)
comparison_rows = []

for model_name, model in candidates.items():
    scores = cross_val_score(
        make_pipeline(model),
        X_development,
        y_development,
        cv=cross_validation,
        scoring="accuracy",
        groups=groups_development,
        n_jobs=1,
    )
    comparison_rows.append(
        {
            "model": model_name,
            "cv_mean_accuracy": scores.mean(),
            "cv_std_accuracy": scores.std(ddof=1),
            "fold_scores": np.round(scores, 3),
        }
    )

model_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values(
        ["cv_mean_accuracy", "cv_std_accuracy"],
        ascending=[False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)
display(model_comparison)

### 4a. One-change experiment: TicketGroupSize

本轮只加入 `TicketGroupSize`（同票号人数），其余模型和验证切分保持不变。票号计数在 Pipeline 内只从当前训练折学习，避免验证信息泄漏。

判断规则：与原逻辑回归基线使用完全相同的家庭分组折；平均 Accuracy 提高且标准差不增大才保留，否则撤回。

In [ ]:
logistic_model = candidates["Logistic Regression"]

baseline_ticket_scores = cross_val_score(
    make_pipeline(logistic_model),
    X_development,
    y_development,
    cv=cross_validation,
    scoring="accuracy",
    groups=groups_development,
    n_jobs=1,
)

ticket_group_scores = cross_val_score(
    make_pipeline(logistic_model, include_ticket_group_size=True),
    X_development,
    y_development,
    cv=cross_validation,
    scoring="accuracy",
    groups=groups_development,
    n_jobs=1,
)

ticket_experiment = pd.DataFrame(
    {
        "方案": ["当前基线", "加入同票号人数"],
        "平均准确率": [baseline_ticket_scores.mean(), ticket_group_scores.mean()],
        "标准差": [baseline_ticket_scores.std(ddof=1), ticket_group_scores.std(ddof=1)],
        "五折成绩": [
            np.round(baseline_ticket_scores, 3),
            np.round(ticket_group_scores, 3),
        ],
    }
)

mean_change = ticket_group_scores.mean() - baseline_ticket_scores.mean()
std_change = ticket_group_scores.std(ddof=1) - baseline_ticket_scores.std(ddof=1)
keep_ticket_group_size = mean_change > 0 and std_change <= 0

display(ticket_experiment)
print(f"平均准确率变化: {mean_change:+.3f}")
print(f"标准差变化: {std_change:+.3f}")
print(f"本轮决定: {'保留' if keep_ticket_group_size else '撤回'}同票号人数特征")

### 4b. One-change experiment: FarePerPersonLog

本轮只增加 `FarePerPersonLog`：先用当前训练折内观察到的同票号人数计算人均票价，再做 `log1p`。原始 `FareLog`、模型和家庭分组切分保持不变。

判断规则仍为：平均 Accuracy 提高且标准差不增大才保留，否则撤回。

In [ ]:
fare_per_person_scores = cross_val_score(
    make_pipeline(logistic_model, include_fare_per_person=True),
    X_development,
    y_development,
    cv=cross_validation,
    scoring="accuracy",
    groups=groups_development,
    n_jobs=1,
)

fare_experiment = pd.DataFrame(
    {
        "方案": ["当前基线", "增加人均票价"],
        "平均准确率": [baseline_ticket_scores.mean(), fare_per_person_scores.mean()],
        "标准差": [baseline_ticket_scores.std(ddof=1), fare_per_person_scores.std(ddof=1)],
        "五折成绩": [
            np.round(baseline_ticket_scores, 3),
            np.round(fare_per_person_scores, 3),
        ],
    }
)

fare_mean_change = fare_per_person_scores.mean() - baseline_ticket_scores.mean()
fare_std_change = fare_per_person_scores.std(ddof=1) - baseline_ticket_scores.std(ddof=1)
keep_fare_per_person = fare_mean_change > 0 and fare_std_change <= 0

display(fare_experiment)
print(f"平均准确率变化: {fare_mean_change:+.3f}")
print(f"标准差变化: {fare_std_change:+.3f}")
print(f"本轮决定: {'保留' if keep_fare_per_person else '撤回'}人均票价特征")

### 4c. One-change experiment: FamilySizeCategory

本轮只增加家庭规模类别：`Alone`（1 人）、`Small`（2–4 人）、`Large`（5 人及以上）。原始 `FamilySize`、模型和家庭分组切分保持不变。

判断规则仍为：平均 Accuracy 提高且标准差不增大才保留，否则撤回。

In [ ]:
family_category_scores = cross_val_score(
    make_pipeline(logistic_model, include_family_size_category=True),
    X_development,
    y_development,
    cv=cross_validation,
    scoring="accuracy",
    groups=groups_development,
    n_jobs=1,
)

family_category_experiment = pd.DataFrame(
    {
        "方案": ["当前基线", "增加家庭规模类别"],
        "平均准确率": [baseline_ticket_scores.mean(), family_category_scores.mean()],
        "标准差": [baseline_ticket_scores.std(ddof=1), family_category_scores.std(ddof=1)],
        "五折成绩": [
            np.round(baseline_ticket_scores, 3),
            np.round(family_category_scores, 3),
        ],
    }
)

family_mean_change = family_category_scores.mean() - baseline_ticket_scores.mean()
family_std_change = family_category_scores.std(ddof=1) - baseline_ticket_scores.std(ddof=1)
keep_family_size_category = family_mean_change > 0 and family_std_change <= 0

display(family_category_experiment)
print(f"平均准确率变化: {family_mean_change:+.3f}")
print(f"标准差变化: {family_std_change:+.3f}")
print(f"本轮决定: {'保留' if keep_family_size_category else '撤回'}家庭规模类别特征")

### 5. Tune only the leading non-dummy candidate

调参范围保持很小，并继续使用同一套家庭分组五折。这样模型选择和调参都不会把同一家人拆到训练与验证两边。

In [ ]:
parameter_grids = {
    "Logistic Regression": {
        "model__C": [0.1, 1.0, 10.0],
    },
    "Random Forest": {
        "model__n_estimators": [200, 400],
        "model__max_depth": [None, 6, 10],
        "model__min_samples_leaf": [1, 2],
    },
    "Gradient Boosting": {
        "model__n_estimators": [50, 100],
        "model__learning_rate": [0.05, 0.10],
        "model__max_depth": [1, 2],
    },
}

selected_model_name = next(
    name for name in model_comparison["model"] if name != "Dummy"
)
print(f"Selected for bounded tuning: {selected_model_name}")

search = GridSearchCV(
    estimator=make_pipeline(candidates[selected_model_name]),
    param_grid=parameter_grids[selected_model_name],
    scoring="accuracy",
    cv=cross_validation,
    n_jobs=1,
    refit=True,
    return_train_score=False,
)
search.fit(X_development, y_development, groups=groups_development)

print(f"Best development CV accuracy: {search.best_score_:.3f}")
print(f"Best parameters: {search.best_params_}")

### 6. Evaluate once on the untouched holdout

验收规则固定为：候选模型的 holdout Accuracy 必须至少比多数类 Dummy 基线高 `0.02`。如果不满足，流程主动停止，不生成提交文件。

In [ ]:
best_development_pipeline = search.best_estimator_
holdout_predictions = best_development_pipeline.predict(X_holdout).astype(int)
holdout_accuracy = accuracy_score(y_holdout, holdout_predictions)

dummy_pipeline = make_pipeline(candidates["Dummy"])
dummy_pipeline.fit(X_development, y_development)
dummy_holdout_predictions = dummy_pipeline.predict(X_holdout).astype(int)
dummy_holdout_accuracy = accuracy_score(y_holdout, dummy_holdout_predictions)
improvement = holdout_accuracy - dummy_holdout_accuracy

holdout_summary = pd.DataFrame(
    {
        "metric": ["Dummy holdout accuracy", "Candidate holdout accuracy", "Improvement"],
        "value": [dummy_holdout_accuracy, holdout_accuracy, improvement],
    }
)
display(holdout_summary.style.format({"value": "{:.3f}"}))

if improvement < 0.02:
    raise RuntimeError(
        "候选模型没有明显优于 Dummy 基线。请检查数据、特征和 Pipeline；不要生成提交文件。"
    )

In [ ]:
fig, axis = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_holdout,
    holdout_predictions,
    display_labels=["Did not survive", "Survived"],
    cmap="Blues",
    colorbar=False,
    ax=axis,
)
axis.set_title(f"Holdout confusion matrix: {selected_model_name}")
plt.tight_layout()
plt.show()

classification_metrics = pd.DataFrame(
    classification_report(
        y_holdout,
        holdout_predictions,
        target_names=["Did not survive", "Survived"],
        output_dict=True,
        zero_division=0,
    )
).T
display(classification_metrics.round(3))

### 7. Inspect model signals carefully

系数绝对值或树模型 importance 只能说明模型在当前特征表示下依赖哪些信号，不能解释为因果影响。

In [ ]:
fitted_preprocessor = best_development_pipeline.named_steps["preprocess"]
fitted_model = best_development_pipeline.named_steps["model"]
feature_names = fitted_preprocessor.get_feature_names_out()

if hasattr(fitted_model, "coef_"):
    signal_strength = np.abs(fitted_model.coef_).ravel()
    signal_label = "absolute coefficient"
elif hasattr(fitted_model, "feature_importances_"):
    signal_strength = fitted_model.feature_importances_
    signal_label = "feature importance"
else:
    signal_strength = None

if signal_strength is not None:
    model_signals = (
        pd.DataFrame({"feature": feature_names, signal_label: signal_strength})
        .sort_values(signal_label, ascending=False)
        .head(15)
    )
    display(model_signals)
else:
    print("The selected estimator does not expose coefficients or feature importances.")

### 8. Refit on all training rows and create the submission

模型选择已结束。现在克隆选中的 Pipeline，在全部训练数据上重新拟合，然后预测官方测试集。

In [ ]:
final_pipeline = clone(search.best_estimator_)
final_pipeline.fit(X, y)
test_predictions = final_pipeline.predict(test).astype(int)

submission = pd.DataFrame(
    {
        "PassengerId": test["PassengerId"].astype(int),
        "Survived": test_predictions,
    }
)
validate_submission(submission, test)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
submission_path = OUTPUT_DIR / "submission.csv"
model_path = MODEL_DIR / "titanic_pipeline.joblib"

submission.to_csv(submission_path, index=False)
joblib.dump(final_pipeline, model_path)

print(f"Saved submission: {submission_path}")
print(f"Saved model: {model_path}")
display(submission.head())

## Checks

最后重新读取磁盘产物并验证：提交结构正确、预测只能为 0/1、模型重新加载后的预测完全一致。

In [ ]:
saved_submission = pd.read_csv(submission_path)
validate_submission(saved_submission, test)

reloaded_pipeline = joblib.load(model_path)
reloaded_predictions = reloaded_pipeline.predict(test).astype(int)
assert np.array_equal(reloaded_predictions, test_predictions)
assert submission_path.stat().st_size > 0
assert model_path.stat().st_size > 0

print("All artifact checks passed.")
print(f"Rows: {len(saved_submission)}")
print(f"Columns: {list(saved_submission.columns)}")
print(f"Prediction values: {sorted(saved_submission['Survived'].unique())}")

## Next Steps

1. 在 Kaggle 比赛页手动上传 `outputs/submission.csv`。
2. 在 README 或实验记录中写下 Kaggle 返回的分数和提交日期；在实际上传前不要猜测分数。
3. 回顾 CV 平均值、CV 波动和 holdout 结果是否一致。若差异很大，优先调查样本量和过拟合，而不是继续追逐公开榜单。
4. 第一版完成后，可以尝试一次受控改动（例如改变 Title 稀有阈值），仍然使用相同的验证协议。